# Selección aleatoria de instancias (baseline)

Este notebook realiza una **selección aleatoria uniforme** de 750,000 instancias a partir de las primeras 2.5M filas del conjunto de entrenamiento.

Sirve como **baseline de comparación** frente a la selección inteligente 80/20 basada en el Oráculo LightGBM (`seleccion_instancias_lightgbm.ipynb`), donde el 80% de las instancias proviene de la zona gris (alta incertidumbre) y el 20% de la zona segura (alta confianza).

Los archivos generados (`X_train_750k_random.npy`, `y_train_750k_random.npy`) tienen nombres distintos a los del muestreo 80/20 para no sobrescribirlos.

In [ ]:
import numpy as np
import gc #liberacion de memoria ram.

In [ ]:
#Carga de datos (mmap: no se carga la matriz completa en RAM)

X_mmap = np.load('../../train_data.npy', mmap_mode='r')[:2500000]

print(f"Dimensiones de variables de entrenamiento: {X_mmap.shape}")

with open('../../train_data.npy', 'rb') as f:
    _ = np.load(f)          # Saltamos la matriz X
    y_completo = np.load(f)  # Cargamos el vector y en memoria

In [ ]:
#Distribucion de clases original (primeras 2.5M filas)
y_original = y_completo[:2500000].ravel().astype(int)
conteo_original = np.bincount(y_original)

print("Distribución de clases original:")
for clase, n in enumerate(conteo_original):
    print(f"   Clase {clase}: {n} ({100 * n / len(y_original):.2f}%)")

In [ ]:
#Seleccion aleatoria uniforme de instancias
N_TOTAL_OBJETIVO = 750000
np.random.seed(42)  # reproducibilidad

idx_finales = np.sort(
    np.random.choice(len(X_mmap), size=N_TOTAL_OBJETIVO, replace=False)
)

print(f"Selección final: {len(idx_finales)} instancias")

In [ ]:
# Extracción física de los datos (solo se materializan las filas seleccionadas)
X_train_AG = X_mmap[idx_finales]
y_train_AG = y_completo[:2500000][idx_finales]

print(f"Dimensiones X: {X_train_AG.shape}, y: {y_train_AG.shape}")

In [ ]:
#Verificacion: la muestra aleatoria debe preservar la proporcion de clases
y_muestra = y_train_AG.ravel().astype(int)
conteo_muestra = np.bincount(y_muestra, minlength=len(conteo_original))

print("Distribución de clases (original vs muestra aleatoria):")
for clase in range(len(conteo_original)):
    p_orig = 100 * conteo_original[clase] / len(y_original)
    p_muestra = 100 * conteo_muestra[clase] / len(y_muestra)
    print(f"   Clase {clase}: original {p_orig:.2f}% | muestra {p_muestra:.2f}% ({conteo_muestra[clase]} instancias)")

In [ ]:
#Guardar archivos para el Algoritmo
np.save('X_train_750k_random.npy', X_train_AG)
np.save('y_train_750k_random.npy', y_train_AG)

print("Archivos guardados: X_train_750k_random.npy, y_train_750k_random.npy")

In [ ]:
#Liberacion de memoria
del X_train_AG, y_train_AG, y_completo, y_original, y_muestra
gc.collect()